# SQL MURDER MISTERY - GROUP 003 🔍 

### Enunciado

<div align="center">
<blockquote>
<p><em>A crime has taken place and the detective needs your help.</em></p>
<p><em>The detective gave you the crime scene report, but you somehow lost it.</em></p>
<p><em>You vaguely remember that the crime was a <strong>murder</strong> that occurred sometime on <strong>Jan.15, 2018</strong> and that it took place in <strong>SQL City</strong>.</em></p>
<p><em>Start by retrieving the corresponding crime scene report from the police department's database.</em></p>
</blockquote>
</div>

## **1. Preparación** ##

In [ ]:
import pandas as pd
import sqlite3
import os

In [3]:
os.getcwd()

'c:\\Users\\sandy\\AppData\\Local\\Programs\\Microsoft VS Code'

In [4]:
connection = sqlite3.connect("C:/users/sandy/Documents/GitHub/Bridge-sgusmar/Team_Challenges/TC_02_Sprint_06_SQL/parte_001/data/sql-murder-mystery.db")
cursor_murder = connection.cursor()

In [5]:
def sql_query(query):

    cursor_murder.execute(query)


    ans = cursor_murder.fetchall()


    names = [description[0] for description in cursor_murder.description]

    return pd.DataFrame(ans,columns=names)

## **2. Inicio de la investigación: Búsqueda de testigos** ##

### Buscamos primero en los registros de crímenes un asesinato cometido el 15 de enero de 2018 en SQL City

In [6]:
query = '''
SELECT * 
FROM crime_scene_report
WHERE city in ("SQL City") AND type in ("murder")
'''
sql_query(query)

,date,type,description,city
0,20180215,murder,REDACTED REDACTED REDACTED,SQL City
1,20180215,murder,Someone killed the guard! He took an arrow to ...,SQL City
2,20180115,murder,Security footage shows that there were 2 witne...,SQL City


In [64]:
cursor_murder.execute(query)
cursor_murder.fetchall()
# 

[(20180215, 'murder', 'REDACTED REDACTED REDACTED', 'SQL City'),
 (20180215,
  'murder',
  'Someone killed the guard! He took an arrow to the knee!',
  'SQL City'),
 (20180115,
  'murder',
  'Security footage shows that there were 2 witnesses. The first witness lives at the last house on "Northwestern Dr". The second witness, named Annabel, lives somewhere on "Franklin Ave".',
  'SQL City')]

### El registro con índice 2 indica que hay 2 testigos: uno que vive en la última casa de "Northwestern Dr" y Annabel en "Franklin Ave". Usamos estos datos para localizar a los testigos.

In [ ]:
query = '''
SELECT * 
FROM person
WHERE address_street_name in ("Northwestern Dr")
ORDER BY address_number DESC
'''
sql_query(query)

,id,name,license_id,address_number,address_street_name,ssn
0,14887,Morty Schapiro,118009,4919,Northwestern Dr,111564949
1,17729,Lasonya Wildey,439686,3824,Northwestern Dr,917817122
2,53890,Sophie Tiberio,957671,3755,Northwestern Dr,442830147
3,73368,Torie Thalmann,773862,3697,Northwestern Dr,341559436
4,96595,Coretta Cubie,303645,3631,Northwestern Dr,378403829
5,19420,Cody Schiel,890431,3524,Northwestern Dr,947110049
6,93509,Emmitt Aceuedo,916706,3491,Northwestern Dr,979073160
7,87456,Leonora Wolfsberger,215868,3483,Northwestern Dr,565203106
8,36378,Freddie Ellzey,267882,3449,Northwestern Dr,474117596
9,53076,Boris Bijou,664914,3327,Northwestern Dr,401191868


### De aquí descubrimos el nombre de uno de los testigos: "Morty Schapiro". Buscamos en la tabla interview su declaración.

In [66]:
query = '''
SELECT * 
FROM interview
WHERE person_id in ("14887")
'''
cursor_murder.execute(query)
cursor_murder.fetchall()

[(14887,
  'I heard a gunshot and then saw a man run out. He had a "Get Fit Now Gym" bag. The membership number on the bag started with "48Z". Only gold members have those bags. The man got into a car with a plate that included "H42W".')]

**Info de la declaración: El hombre que escapó tenía una membresía de oro que empieza por 48Z y se metió en un coche con una matrícula que contenía H42W**

### Buscamos al segundo testigo: Annabel

In [67]:
query = '''
SELECT * 
FROM person
WHERE address_street_name in ("Franklin Ave")
AND name LIKE ("%Annabel%")
'''
cursor_murder.execute(query)
cursor_murder.fetchall()

[(16371, 'Annabel Miller', 490173, 103, 'Franklin Ave', 318771143)]

**Descubrimos que el segundo testigo es Annabel Miller**

In [68]:
query = '''
SELECT * 
FROM interview
WHERE person_id in ("16371")
'''
cursor_murder.execute(query)
cursor_murder.fetchall()

[(16371,
  'I saw the murder happen, and I recognized the killer from my gym when I was working out last week on January the 9th.')]

**Info de la declaración: El asesino entrenó la semana pasada, el 9 de enero**

## Análisis de las pistas #

1. Matrícula del coche

In [8]:
query = '''
SELECT * 
FROM drivers_license
WHERE plate_number LIKE ("%H42W%")
'''
sql_query(query)

,id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model
0,183779,21,65,blue,blonde,female,H42W0X,Toyota,Prius
1,423327,30,70,brown,brown,male,0H42W2,Chevrolet,Spark LS
2,664760,21,71,black,black,male,4H42WR,Nissan,Altima


2. Membresía gimnasio

In [36]:
query = """
SELECT p.*
FROM drivers_license as d
JOIN person as p ON p.license_id = d.id
WHERE d.plate_number LIKE '%H42W%'
"""
sql_query(query)

,id,name,license_id,address_number,address_street_name,ssn
0,51739,Tushar Chandra,664760,312,Phi St,137882671
1,67318,Jeremy Bowers,423327,530,"Washington Pl, Apt 3A",871539279
2,78193,Maxine Whitely,183779,110,Fisk Rd,137882671


In [44]:
query = """
SELECT *
FROM get_fit_now_member
WHERE name in ("Jeremy Bowers", "Tushar Chandra", "Maxine Whitely") 
AND membership_status in ("gold")
"""
sql_query(query)

,id,person_id,name,membership_start_date,membership_status
0,48Z55,67318,Jeremy Bowers,20160101,gold


In [43]:
query = """
SELECT *
FROM get_fit_now_check_in
WHERE membership_id in ("48Z55")
"""
sql_query(query)

,membership_id,check_in_date,check_in_time,check_out_time
0,48Z55,20180109,1530,1700


**La única persona con membresía "gold" que entrenó el día 9 de enero de 2018 de las 3 personas sospechosas por la matrícula de su coche es Jeremy Bowers**

![Captura de pantalla de la solución 1 en la página web de SQL Murder Mistery](solucion_1.png)

## **Parte extra** 

### Buscamos la declaración del asesino Jeremy Bowers

In [49]:
query = """
SELECT *
FROM person
WHERE name in ("Jeremy Bowers")
"""
sql_query(query)

,id,name,license_id,address_number,address_street_name,ssn
0,67318,Jeremy Bowers,423327,530,"Washington Pl, Apt 3A",871539279


In [52]:
query = """
SELECT *
FROM interview
WHERE person_id in ("67318")
"""
cursor_murder.execute(query)
cursor_murder.fetchall()

[(67318,
  'I was hired by a woman with a lot of money. I don\'t know her name but I know she\'s around 5\'5" (65") or 5\'7" (67"). She has red hair and she drives a Tesla Model S. I know that she attended the SQL Symphony Concert 3 times in December 2017.\n')]

### Sabemos la altura, el género, el color de pelo y la marca y modelo del coche. Recurrimos a la tabla drivers_license para continuar con la investigación

In [77]:
query = """
SELECT *
FROM drivers_license
WHERE gender = ("female")
AND hair_color in ("red")
AND car_make in ("Tesla")
AND (height = "65" OR height < 68)
"""
sql_query(query)

,id,age,height,eye_color,hair_color,gender,plate_number,car_make,car_model
0,202298,68,66,green,red,female,500123,Tesla,Model S
1,291182,65,66,blue,red,female,08CM64,Tesla,Model S
2,918773,48,65,black,red,female,917UU3,Tesla,Model S


### Hay 3 posibles resultados, busquemos ids a los que corresponden en la tabla "person" para poder ver quién asistió 3 veces al evento SQL Symphony Concert

In [86]:
query = """
SELECT p.*
FROM drivers_license as d
JOIN person as p ON p.license_id = d.id
WHERE d.id LIKE ("202298")
"""
sql_query(query)

,id,name,license_id,address_number,address_street_name,ssn
0,99716,Miranda Priestly,202298,1883,Golden Ave,987756388


In [87]:
query = """
SELECT p.*
FROM drivers_license as d
JOIN person as p ON p.license_id = d.id
WHERE d.id LIKE ("291182")
"""
sql_query(query)

,id,name,license_id,address_number,address_street_name,ssn
0,90700,Regina George,291182,332,Maple Ave,337169072


In [88]:
query = """
SELECT p.*
FROM drivers_license as d
JOIN person as p ON p.license_id = d.id
WHERE d.id LIKE ("918773")
"""
sql_query(query)

,id,name,license_id,address_number,address_street_name,ssn
0,78881,Red Korb,918773,107,Camerata Dr,961388910


**Tres personas coinciden con la descripción: Regina George, Red Korb y Miranda Priestly. Confirmemos ahora el dato de la asistencia al concierto**

In [96]:
query = """
SELECT p.*
FROM facebook_event_checkin as f
JOIN person as p ON p.id = f.person_id
WHERE event_name in ("SQL Symphony Concert")
AND (person_id = "78881" OR person_id = "90700" OR person_id = "99716")
"""
sql_query(query)

,id,name,license_id,address_number,address_street_name,ssn
0,99716,Miranda Priestly,202298,1883,Golden Ave,987756388
1,99716,Miranda Priestly,202298,1883,Golden Ave,987756388
2,99716,Miranda Priestly,202298,1883,Golden Ave,987756388


**La artífice es Miranda Priestly**

![Captura de pantalla de la segunda parte de la solución en la página web de SQL Murder Mistery](solucion_2.png)